### 👥 Group Members Information

| Member | Student Name | Registration / Student ID |
| :--- | :--- | :--- |
| **Student 1** | | |
| **Student 2** | | |
| **Student 3** | | |

**Date:** ____________________

---

# Lecture 5 · Guided practical: building and exploring a Kigali feature set

**Machine Learning for Earth Observation · Year 3**

You will build the feature sets used in Lecture 5 from the real Kigali Sentinel-2 L2A data and test them with the spatial-block cross-validation of Lecture 4.

| Step | What you do | Learning objectives | Time |
|---|---|---|---|
| 1 | Compute 8 spectral indices safely | bands and indices (LO2, LO9) | 5 min |
| 2 | Extract features at the training pixels; box plots per class | role of features (LO1, LO2) | 5 min |
| 3 | Focal statistics| spatial / texture features (LO3) | 6 min |
| 4 | Correlation matrix and a correlation filter | redundancy, selection (LO6, LO7) | 5 min |
| 5 | kNN and random forest with and without standardisation | scaling (LO5) | 4 min |
| 6 | PCA: scree plot, loadings, PC images | dimensionality reduction (LO8) | 6 min |
| 7 | Compare feature sets; permutation importance | selection (LO6, LO7) | 8 min |
| 8 | Temporal features from January, May and September 2026 | temporal features (LO4, LO9) | 6 min |
| 9 | Forward feature selection with spatial CV | selection (LO7) | 10 min |

**Data** (folder `ml4eo/data/`, opened with rasterio):
* the 12 Sentinel-2 L2A bands of **3 September 2026** on one 20 m grid (2380 × 1702 pixels, UTM zone 36S), stored as reflectance (0–1), plus the Scene Classification Layer (SCL);
* the same bands and SCL of **21 January 2026** and **13 May 2026**, on the same grid (we use B02, B03, B04, B08 and B11);
* `kigali_training_polygons.csv`: the 1,225 labelled polygon pixels of Lectures 3–4 (49 polygons, 5 classes) with row/column, polygon ID and a 5 km spatial-block ID.

⚠️ The class labels were derived from the September image itself (Lecture 3). Absolute accuracies are optimistic; use them to compare feature sets.

Complete the cells marked **`# TODO`** (replace `...`) and answer the **Questions**. Run the cells in order.

## 0. Set-up

Only numpy, pandas, matplotlib, scipy, scikit-image and scikit-learn are needed, plus rasterio for reading the images.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import uniform_filter

from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.decomposition import PCA
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.metrics import accuracy_score, f1_score
from sklearn.inspection import permutation_importance
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)

import rasterio

: 

In [ ]:
import os
def find_data_dir():
    for candidate in ["../data", "../../data", "data", "ML4EO/data"]:
        if os.path.exists(candidate) and os.path.isdir(candidate):
            return candidate
    return "../data"

DATA = find_data_dir()                   # dynamically locate the data folder

def load_band(date, band):
    # Open one Sentinel-2 band of one date: reflectance 0-1, NaN where there is no data (value 0)
    path = f"{DATA}/{date}-00_00_{date}-23_59_Sentinel-2_L2A_{band}_(Raw).tiff"
    with rasterio.open(path) as src:
        a = src.read(1)
    a[a == 0] = np.nan
    return a

SCL_COLOURS = {0: (0, 0, 0), 1: (255, 0, 0), 2: (47, 47, 47), 3: (100, 50, 0), 4: (0, 160, 0), 5: (255, 230, 90),
               6: (0, 0, 255), 7: (128, 128, 128), 8: (192, 192, 192), 9: (255, 255, 255), 10: (100, 200, 255), 11: (255, 150, 255)}

def read_scl(date):
    # The scene classification is stored as a coloured picture: turn each colour back into its class code
    path = f"{DATA}/{date}-00_00_{date}-23_59_Sentinel-2_L2A_Scene_classification_map_.tiff"
    with rasterio.open(path) as src:
        rgb = np.round(src.read() * 255)             # 3 layers of colour values 0-255
    codes = np.zeros(rgb.shape[1:], dtype=np.uint8)   # 0 = no data
    for code_, (r, g_, b) in SCL_COLOURS.items():
        codes[(rgb[0] == r) & (rgb[1] == g_) & (rgb[2] == b)] = code_
    return codes


In [ ]:
BANDS = ["B01", "B02", "B03", "B04", "B05", "B06", "B07", "B08", "B8A", "B09", "B11", "B12"]

s = {b: load_band("2026-09-03", b) for b in BANDS}       # September reflectance, band by band
refl = np.stack([s[b] for b in BANDS])                   # (12 bands, rows, cols)

print("bands:", refl.shape, refl.dtype)
print ("reflectance range: ", round(float(np.nanmin(refl)), 4), "-", round(float(np.nanmax(refl)), 4))

In [ ]:
scl = read_scl("2026-09-03")
H, W = scl.shape
valid = np.isin(scl, [4, 5, 6]) & np.all(np.isfinite(refl), axis=0)   # SCL: vegetation, not vegetated, water
print(f"valid pixels: {100 * valid.mean():.2f}%")

In [ ]:
poly = pd.read_csv(f"{DATA}/kigali_training_polygons.csv")
print(poly.shape); print(poly["class"].value_counts())
CLASSES = ["water", "dense vegetation", "sparse vegetation", "built-up", "bare soil"]
CC = {"water": "#2A7FA8", "dense vegetation": "#2E6B3F", "sparse vegetation": "#A8C66C", "built-up": "#B5503C", "bare soil": "#9A8468"}
def stretch(a, lo=2, hi=98):
    l, h = np.nanpercentile(a, [lo, hi]); return np.clip((a - l) / (h - l), 0, 1)
rgb = np.dstack([stretch(s["B04"]), stretch(s["B03"]), stretch(s["B02"])])

⚠️ **Data types.** These files store reflectance as `float32`. Many archives (Sentinel-2 SAFE files, Google Earth Engine) instead deliver integers, **DN = reflectance × 10,000**, as `uint16`. With unsigned integers, a subtraction that should give a negative number wraps around. Always convert such data to float (and to reflectance) before computing indices.

## Step 1 · Spectral indices  *(LO2, LO9)*

Write a safe normalised-difference function and compute eight indices (formulas as in the lecture).

In [ ]:
def norm_diff(a, b):
    """(a - b) / (a + b); NaN where the denominator is (almost) zero."""
    den = a + b
    out = np.full(a.shape, np.nan, dtype="float32")
    # divide (a - b) by den into `out`, but only where |den| > 1e-6 (hint: np.divide(..., out=out, where=...))
    np.divide(a - b, den, out=out, where=np.abs(den) > 1e-6)
    return out

IX = {
    "NDVI":  norm_diff(s["B08"], s["B04"]),
    "NDRE":  norm_diff(s["B08"], s["B05"]),
    "SAVI":  1.5 * (s["B08"] - s["B04"]) / (s["B08"] + s["B04"] + 0.5),
    "EVI":   2.5 * (s["B08"] - s["B04"]) / (s["B08"] + 6 * s["B04"] - 7.5 * s["B02"] + 1),
    "NDWI":  norm_diff(s["B03"], s["B08"]),
    "MNDWI": norm_diff(s["B03"], s["B11"]),
    "NDBI":  norm_diff(s["B11"], s["B08"]),
    "BSI":   norm_diff(s["B11"] + s["B04"], s["B08"] + s["B02"]),
}
INDICES = list(IX)

def masked(a):
    a = a.astype(float).copy(); a[~valid] = np.nan; return a

fig, ax = plt.subplots(2, 3, figsize=(15, 6.6))
panels = [("True colour", None, None), ("NDVI", "NDVI", "YlGn"), ("NDRE", "NDRE", "YlGn"),
          ("MNDWI", "MNDWI", "Blues"), ("NDBI", "NDBI", "OrRd"), ("BSI", "BSI", "copper_r")]
for a, (t, k, cm) in zip(ax.ravel(), panels):
    if k is None: a.imshow(rgb)
    else:
        d = masked(IX[k]); lo, hi = np.nanpercentile(d, [2, 98])
        plt.colorbar(a.imshow(d, cmap=cm, vmin=lo, vmax=hi), ax=a, fraction=0.035)
    a.set_title(t); a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

## Step 2 · Features at the training pixels  *(LO1, LO2)*

Each training pixel becomes one row of the ML table; each band or index becomes one column.

In [ ]:
r, c = poly["row"].values, poly["col"].values
X = poly.copy()
for b in BANDS:
    X[b] = s[b][r, c]
for k, v in IX.items():
    X[k] = v[r, c]
y = X["class"].values

fig, ax = plt.subplots(1, 4, figsize=(17, 3.6))
for a, f in zip(ax, ["NDVI", "MNDWI", "NDBI", "BSI"]):
    bp = a.boxplot([X.loc[X["class"] == k, f] for k in CLASSES], orientation="horizontal", patch_artist=True, widths=0.6)
    for p, k in zip(bp["boxes"], CLASSES): p.set_facecolor(CC[k])
    a.set_yticks(range(1, 6)); a.set_yticklabels(CLASSES if f == "NDVI" else [""] * 5); a.set_title(f)
plt.tight_layout(); plt.show()
X.groupby("class")[["NDVI", "MNDWI", "NDBI", "BSI"]].median().round(3)

**Question 1.** Which index separates water best from the other classes? Which pair of classes does *no* index separate? Use the box plots and the medians.

*Your answer:*

- **Best index for water:** **MNDWI** (Modified Normalised Difference Water Index) separates water best: its median for water is **+0.40**, whereas all four terrestrial classes have strongly negative medians between −0.40 and −0.54 with complete separation in the box plots. (NDVI also separates water with a negative median of −0.14 vs positive values for all land classes).
- **Inseparable class pair:** **Bare soil and built-up** are not separated by any of the four spectral indices. Their medians heavily overlap across all indices: NDVI (0.097 vs 0.083), NDBI (0.178 vs 0.163), and BSI (0.263 vs 0.218), showing near-identical distributions due to shared mineral, clay, and roof materials.

## Step 3 · Spatial and texture features  *(LO3)*

**Focal statistics** summarise a moving window around each pixel. The standard deviation follows from two moving averages: std = √(mean(a²) − mean(a)²).

In [ ]:
def focal_mean(a, k):
    return uniform_filter(np.nan_to_num(a), size=k, mode="reflect")

def focal_std(a, k):
    m, m2 = focal_mean(a, k), focal_mean(np.nan_to_num(a) ** 2, k)
    return np.sqrt(np.clip(m2 - m ** 2, 0, None))

SP = {"NDVI_mean5": focal_mean(IX["NDVI"], 5), "NDVI_mean11": focal_mean(IX["NDVI"], 11),
      "NDVI_std5": focal_std(IX["NDVI"], 5), "B08_std5": focal_std(s["B08"], 5)}

zoom = (slice(450, 880), slice(300, 940))            # central and northern Kigali
fig, ax = plt.subplots(1, 4, figsize=(17, 3.3))
ax[0].imshow(rgb[zoom]); ax[0].set_title("True colour")
for a, (t, k, cm) in zip(ax[1:], [("NDVI", None, "YlGn"), ("Focal mean NDVI 11 × 11", "NDVI_mean11", "YlGn"), ("Focal std NDVI 5 × 5", "NDVI_std5", "magma")]):
    d = masked(IX["NDVI"] if k is None else SP[k])[zoom]; lo, hi = np.nanpercentile(d, [2, 98])
    plt.colorbar(a.imshow(d, cmap=cm, vmin=lo, vmax=hi), ax=a, fraction=0.035); a.set_title(t)
for a in ax: a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

In [ ]:
for k, v in SP.items():
    X[k] = v[r, c]

TEXTURE = list(SP) 
X.groupby("class")[["NDVI_std5", "B08_std5"]].median().round(3)

## Step 4 · Redundancy: correlation matrix and a correlation filter  *(LO6, LO7)*

First the 20 spectral features (12 bands + 8 indices) over 200,000 random valid pixels of the whole scene.

In [ ]:
rng = np.random.default_rng(0)
vr, vc = np.nonzero(valid)
pick = rng.choice(len(vr), 200_000, replace=False)
scene = pd.DataFrame({**{b: s[b][vr[pick], vc[pick]] for b in BANDS}, **{k: v[vr[pick], vc[pick]] for k, v in IX.items()}})
corr = scene.corr()

fig, a = plt.subplots(figsize=(7.5, 6.5))
im = a.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1); plt.colorbar(im, ax=a, fraction=0.045)
a.set_xticks(range(20)); a.set_xticklabels(corr.columns, rotation=90); a.set_yticks(range(20)); a.set_yticklabels(corr.index)
plt.show()

pairs = [(a_, b_, corr.loc[a_, b_]) for i, a_ in enumerate(corr.columns) for b_ in corr.columns[i + 1:]]
high = sorted([p for p in pairs if abs(p[2]) > 0.95], key=lambda t: -abs(t[2]))
print(f"{len(pairs)} pairs; |r| > 0.9: {sum(abs(p[2]) > 0.9 for p in pairs)}; |r| > 0.95: {len(high)}")
for a_, b_, r_ in high: print(f"  {a_:5s} – {b_:5s}  r = {r_:.3f}")
print(f"B04 – B08: r = {corr.loc['B04', 'B08']:.3f}")

Now a **correlation filter** on the 24 training features: keep features in the listed order, dropping any feature correlated at |r| > 0.95 with a feature already kept.

In [ ]:
ALL = BANDS + INDICES + TEXTURE
tcorr = X[ALL].corr().abs()
kept = []
for f in ALL:
    if all(tcorr.loc[f, k] <= 0.95 for k in kept):
        kept.append(f)
print(f"kept {len(kept)} of {len(ALL)}:", kept)
print("dropped:", [f for f in ALL if f not in kept])

**Question 2.** Which feature pairs have |r| > 0.95? Explain two of them physically. Why is the correlation of B04 and B08 so low, and why does that make NDVI useful?

*Your answer:*

- **Highly correlated pairs ($|r| > 0.95$):** 13 pairs exceed 0.95, including SAVI–EVI ($r = 0.995$), NDVI–NDRE ($r = 0.987$), B07–B8A ($r = 0.981$), B02–B03 ($r = 0.966$), B04–B05 ($r = 0.965$), and NDBI–BSI ($r = 0.961$).
- **Physical explanations:**
  1. *B07–B8A ($r = 0.981$):* Both are adjacent bands located on the near-infrared vegetation plateau, where reflectance is governed by scattering in the spongy mesophyll layer of plant leaves.
  2. *B02–B03 ($r = 0.966$):* Blue and green visible bands respond similarly across soils, dry vegetation, and roofs, dominated by broad surface albedo.
- **B04–B08 correlation ($r = 0.28$) and NDVI utility:** Red (B04) and NIR (B08) have low correlation because photosynthetic vegetation drives them in opposite directions: chlorophyll strongly absorbs red light (low B04), while internal leaf cellular structure strongly reflects NIR (high B08). Because the two bands provide complementary, orthogonal information, normalized difference (NDVI) combines them into a high-contrast metric that sharply discriminates vegetation from non-vegetated surfaces.

## Step 5 · Scaling: kNN vs random forest  *(LO5)*

All models are evaluated with **5-fold spatial-block cross-validation** (5 km blocks, Lecture 4). To mimic raw L2A files, the bands are used as integers (DN) together with the indices (−1 to 1).

In [ ]:
cv = list(GroupKFold(5).split(X, y, X["block_id"]))

def evaluate(model, feats, df=X):
    pred = cross_val_predict(model, df[feats].values, y, cv=cv)
    return {"OA": accuracy_score(y, pred), "macro F1": f1_score(y, pred, average="macro"),
            "F1 bare soil": f1_score(y == "bare soil", pred == "bare soil"), "F1 built-up": f1_score(y == "built-up", pred == "built-up")}

X_dn = X.copy()
X_dn[BANDS] = X_dn[BANDS] * 10000                    # bands in DN, indices unchanged
FEATS = BANDS + INDICES
rf = RandomForestClassifier(n_estimators=300, random_state=0, n_jobs=-1)
scaling = pd.DataFrame({
    "kNN raw":          evaluate(KNeighborsClassifier(5), FEATS, X_dn),
    "kNN standardised": evaluate(make_pipeline(StandardScaler(), KNeighborsClassifier(5)), FEATS, X_dn),
    "RF raw":           evaluate(rf, FEATS, X_dn),
    "RF standardised":  evaluate(make_pipeline(StandardScaler(), rf), FEATS, X_dn),
}).T.round(3)
scaling

**Question 3.** Why does kNN change with standardisation but the random forest does not? Why must the scaler be inside the pipeline rather than fitted on all data first?

*Your answer:*

- **Why kNN changes with standardization:** kNN computes distances in Euclidean feature space. In raw data, digital numbers (DN) span 0 to 10,000, whereas normalized indices span −1 to +1. The DN scale completely dominates the distance metric, rendering the 8 indices practically invisible. Standardizing scales every feature to zero mean and unit variance, allowing all features to participate equally in neighbor distance calculation.
- **Why Random Forest does not change:** Decision trees evaluate one feature at a time using threshold split rules (e.g., $x_j > \theta$). Any strictly monotonic transformation (like standard scaling) preserves the rank order of the data points, resulting in identical split points, tree topologies, and final predictions.
- **Why the scaler must be inside the pipeline:** Fitting the scaler on the full dataset before cross-validation exposes validation fold statistics (mean and standard deviation) to the model, introducing data leakage. Placing the scaler inside a `Pipeline` guarantees it is fitted strictly on each training fold and only applied to transform the held-out test fold.

## Step 6 · Principal component analysis  *(LO8)*

PCA of the 12 standardised bands, fitted on the 200,000 scene pixels of Step 4 (no labels involved).

In [ ]:
scaler = StandardScaler().fit(scene[BANDS].values)
pca = PCA().fit(scaler.transform(scene[BANDS].values))
evr = pca.explained_variance_ratio_
print("explained variance (%):", np.round(100 * evr[:5], 1), "| cumulative PC1–3:", round(100 * evr[:3].sum(), 1))

fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
ax[0].bar(range(1, 13), 100 * evr, color="#3B6E6A"); ax[0].plot(range(1, 13), 100 * np.cumsum(evr), "o-", color="#E0A526")
ax[0].set_xlabel("component"); ax[0].set_ylabel("% variance")
for k, col in zip(range(3), ["black", "#2E6B3F", "#B5503C"]):
    ax[1].plot(BANDS, pca.components_[k], "o-", color=col, label=f"PC{k + 1}")
ax[1].axhline(0, color="grey", lw=0.8); ax[1].legend(); ax[1].set_ylabel("loading")
plt.tight_layout(); plt.show()

full = np.stack([np.nan_to_num(s[b]) for b in BANDS], axis=-1).reshape(-1, 12)
# standardise `full` with the fitted scaler, project it with the fitted PCA, keep the first 3 components
scores = pca.transform(scaler.transform(full))[:, :3].reshape(H, W, 3)
scores = scores.reshape(H, W, 3)
scores[~valid] = np.nan
fig, ax = plt.subplots(1, 4, figsize=(17, 3.2))
for k in range(3):
    ax[k].imshow(scores[:, :, k], cmap="gray", vmin=np.nanpercentile(scores[:, :, k], 2), vmax=np.nanpercentile(scores[:, :, k], 98)); ax[k].set_title(f"PC{k + 1}")
comp = np.dstack([stretch(scores[:, :, k]) for k in range(3)]); comp[np.isnan(comp)] = 1
ax[3].imshow(comp); ax[3].set_title("PC1–3 as RGB")
for a in ax: a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()
pd.DataFrame(pca.components_[:3], index=["PC1", "PC2", "PC3"], columns=BANDS).round(2)

**Question 4.** What do PC1 and PC2 represent physically (look at the signs of the loadings)? How many components would you keep, and why? Why is PCA not guaranteed to keep the information that separates the classes?

*Your answer:*

- **Physical interpretation of components:**
  - **PC1 (64.5% of variance):** Exhibits uniform positive loadings across all 12 bands (0.24 to 0.33), capturing overall scene **surface brightness / albedo**.
  - **PC2 (26.1% of variance):** Contrasts red-edge and NIR bands (positive loadings, +0.25 to +0.37) against visible and SWIR bands (negative loadings), representing **vegetation greenness** (analogous to a spectral vegetation index).
  - **PC3 (5.3% of variance):** Contrasts SWIR against coastal/blue bands, sensitive to soil exposure and moisture.
- **How many components to keep:** Retaining **3 components** is optimal because they account for **95.9%** of total scene variance, and the scree plot flattens sharply (elbow point) after PC3.
- **Why PCA is not guaranteed to preserve class separability:** PCA is an unsupervised dimensionality reduction method that identifies axes of maximum global variance, without knowledge of class labels. Subtle spectral distinctions necessary to separate two specific classes (e.g., distinguishing bare soil from built-up structures) may lie along lower-variance principal components that get discarded.

## Step 7 · Which features help?  *(LO6, LO7)*

Same labels, same spatial folds, same random forest: only the feature set changes. (This cell takes about a minute.)

In [ ]:
SETS = {
    "RGB (3)": ["B02", "B03", "B04"],
    "10 m bands (4)": ["B02", "B03", "B04", "B08"],
    "12 bands": BANDS,
    "8 indices only*": INDICES,
    "12 bands + 8 indices": BANDS + INDICES,
    "+ spatial/texture (27)": BANDS + INDICES + TEXTURE,
    "correlation filter": kept,
}
res = {k: evaluate(rf, v) for k, v in SETS.items()}
res["3 PCs of 12 bands"] = evaluate(make_pipeline(StandardScaler(), PCA(3), rf), BANDS)   # PCA fitted inside each fold
fs = pd.DataFrame(res).T.round(3); fs["n"] = [len(v) for v in SETS.values()] + [3]
fs

In [ ]:
ax = fs[["OA", "F1 bare soil"]].plot(kind="barh", figsize=(9, 4.5), color=["#3B6E6A", "#9A8468"])
ax.invert_yaxis(); ax.set_xlim(0.6, 0.97); ax.set_xlabel("spatial-block CV"); plt.show()

**Permutation importance** on held-out blocks: how much does accuracy drop when one feature is shuffled, *given all the others*?

In [ ]:
perm = np.zeros(len(ALL))
for tr, te in cv:
    m = RandomForestClassifier(n_estimators=200, random_state=0, n_jobs=-1).fit(X[ALL].values[tr], y[tr])
    perm += permutation_importance(m, X[ALL].values[te], y[te], n_repeats=5, random_state=0).importances_mean / len(cv)
imp_perm = pd.Series(perm, index=ALL).sort_values(ascending=False)
imp_gini = pd.Series(RandomForestClassifier(n_estimators=300, random_state=0, n_jobs=-1).fit(X[ALL], y).feature_importances_, index=ALL).sort_values(ascending=False)
pd.DataFrame({"impurity (top 10)": imp_gini.index[:10], "value": imp_gini.values[:10].round(3),
              "permutation (top 10)": imp_perm.index[:10], "drop in OA": imp_perm.values[:10].round(4)})

In [ ]:
print("permutation importance of the vegetation indices:", imp_perm[["NDVI", "SAVI", "EVI", "NDRE"]].round(4).to_dict())

**Question 5.** (a) Which step in feature design gave the biggest gain? Did texture help? (b) The correlation filter keeps about half of the features: what happens to accuracy? (c) Why is the permutation importance of NDVI close to zero although NDVI is one of the most informative features?

*Your answer:*

- **(a) Biggest gain & texture contribution:**
  - Moving from **RGB (3 bands, OA ≈ 0.76)** to **10 m bands (+B08, OA ≈ 0.89)** and the full **12 bands (OA ≈ 0.91)** provided the largest performance leap (+15 percentage points), primarily by introducing NIR and SWIR bands.
  - Adding spatial/texture features (focal mean and std) yielded negligible gain on this dataset, as the reference polygons are relatively homogeneous and already capture localized spatial spectral responses.
- **(b) Impact of correlation filter:** Filtering features at $|r| > 0.95$ reduced the feature count from 27 down to 13 (a ~50% reduction), while accuracy remained virtually identical (OA ≈ 0.91). This confirms that redundant, highly collinear features add model complexity without contributing new discriminatory power.
- **(c) Low permutation importance of NDVI:** Permutation importance measures the drop in accuracy when a single feature is randomly shuffled *while keeping all other features intact*. Because multiple highly correlated vegetation indices (SAVI, EVI, NDRE with $r > 0.97$) are present in the model, shuffling NDVI has minimal impact because the decision trees can seamlessly split on substitute collinear features. Permutation importance tends to disperse importance across collinear feature groups.

## Step 8 · Temporal features: January, May and September 2026  *(LO4, LO9)*

The labels describe **3 September 2026** (dry season). What do the same places look like on **21 January** (Season A) and **13 May** (Season B)?

In [ ]:
B5 = ["B02", "B03", "B04", "B08", "B11"]          # the five bands we compare across dates
DATES = {"jan": "2026-01-21", "may": "2026-05-13"}
T = {"sep": {b: s[b] for b in B5}}
T["sep"]["scl"] = scl
for d in ["jan", "may"]:
    T[d] = {b: load_band(DATES[d], b) for b in B5}  # same grid as September; reflectance, NaN = no data
    T[d]["scl"] = read_scl(DATES[d])
for d in T:
    T[d]["NDVI"] = norm_diff(T[d]["B08"], T[d]["B04"])
    T[d]["MNDWI"] = norm_diff(T[d]["B03"], T[d]["B11"])
    T[d]["NDBI"] = norm_diff(T[d]["B11"], T[d]["B08"])
    T[d]["valid"] = np.isin(T[d]["scl"], [4, 5, 6]) & np.isfinite(T[d]["NDVI"])
inside = np.isfinite(T["jan"]["B04"])
all3 = T["sep"]["valid"] & T["jan"]["valid"] & T["may"]["valid"]
print(f"Jan/May have data on {100 * inside.mean():.1f}% of the grid; {100 * all3[inside].mean():.1f}% of that is valid on all dates")
print("median NDVI:", {d: round(float(np.nanmedian(T[d]["NDVI"][all3])), 2) for d in T})
nv = all3 & (scl == 5)
green = (T["jan"]["NDVI"] > 0.5) | (T["may"]["NDVI"] > 0.5)
print(f"Sept 'not vegetated' pixels with NDVI > 0.5 in Jan or May: {100 * green[nv].mean():.1f}%")

rr, cc = np.nonzero(inside); box = (slice(rr.min(), rr.max() + 1), slice(cc.min(), cc.max() + 1))
fig, ax = plt.subplots(1, 4, figsize=(17, 3.6))
for a, d, t in zip(ax, ["jan", "may", "sep"], ["NDVI 21 Jan 2026", "NDVI 13 May 2026", "NDVI 3 Sept 2026"]):
    v = T[d]["NDVI"].copy(); v[~all3] = np.nan; a.imshow(v[box], cmap="YlGn", vmin=0, vmax=0.8); a.set_title(t)
dv = T["jan"]["NDVI"] - T["sep"]["NDVI"]; dv[~all3] = np.nan
plt.colorbar(ax[3].imshow(dv[box], cmap="BrBG", vmin=-0.5, vmax=0.5), ax=ax[3], fraction=0.04); ax[3].set_title("NDVI Jan − Sept")
for a in ax: a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

Now the training polygons: keep the pixels that are valid on all three dates, then plot the NDVI trajectory of each class and of each bare-soil and built-up polygon.

In [ ]:
keep = all3[r, c]
Xt = X.loc[keep, ["obs_id", "group_id", "class", "row", "col", "block_id"]].reset_index(drop=True)
rt, ct = Xt["row"].values, Xt["col"].values
for d in T:
    for f in B5 + ["NDVI", "MNDWI", "NDBI"]:
        Xt[f"{f}_{d}"] = T[d][f][rt, ct]
yt = Xt["class"].values
print(f"{len(Xt)} pixels, {Xt['group_id'].nunique()} polygons, classes: {Xt['class'].value_counts().to_dict()}")

dates, lab = ["jan", "may", "sep"], ["21 Jan", "13 May", "3 Sept"]
fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
for k in ["dense vegetation", "sparse vegetation", "bare soil", "built-up"]:
    q = Xt.loc[Xt["class"] == k, [f"NDVI_{d}" for d in dates]].quantile([0.25, 0.5, 0.75]).values
    ax[0].fill_between(range(3), q[0], q[2], color=CC[k], alpha=0.2); ax[0].plot(range(3), q[1], "o-", color=CC[k], label=k)
ax[0].legend(fontsize=8); ax[0].set_title("Class median NDVI (IQR)")
pm = Xt[Xt["class"].isin(["bare soil", "built-up"])].groupby(["class", "group_id"])[[f"NDVI_{d}" for d in dates]].median()
for (k, g), row in pm.iterrows():
    ax[1].plot(range(3), row.values, "o-", color=CC[k], lw=1)
    if row.values[:2].max() > 0.3: ax[1].text(1.02, row.values[1], f"{g} ({k})", fontsize=9)
ax[1].set_title("Each bare-soil and built-up polygon (median)")
for a in ax: a.set_xticks(range(3)); a.set_xticklabels(lab); a.set_ylim(-0.05, 0.85); a.set_ylabel("NDVI")
plt.tight_layout(); plt.show()
pm[pm.max(axis=1) > 0.3].round(2)

Finally, compare single-date and multi-date features on these pixels (spatial-block CV on the blocks that remain).

In [ ]:
cv_t = list(GroupKFold(5).split(Xt, yt, Xt["block_id"]))
F8 = lambda d: [f"{f}_{d}" for f in B5 + ["NDVI", "MNDWI", "NDBI"]]
nd3 = Xt[[f"NDVI_{d}" for d in dates]].values
Xt["NDVI_max"], Xt["NDVI_min"], Xt["NDVI_mean"] = nd3.max(1), nd3.min(1), nd3.mean(1)
Xt["NDVI_amp"] = Xt["NDVI_max"] - Xt["NDVI_min"]
TSETS = {"Sept only (8)": F8("sep"), "Jan only (8)": F8("jan"), "May only (8)": F8("may"),
         "Sept + NDVI max/min/mean/amp (12)": F8("sep") + ["NDVI_max", "NDVI_min", "NDVI_mean", "NDVI_amp"],
         "3 dates stacked (24)": F8("sep") + F8("jan") + F8("may")}
rows = {}
for k, f in TSETS.items():
    p = cross_val_predict(rf, Xt[f].values, yt, cv=cv_t)
    rows[k] = {"OA": accuracy_score(yt, p), "F1 bare soil": f1_score(yt == "bare soil", p == "bare soil"), "F1 built-up": f1_score(yt == "built-up", p == "built-up")}
pd.DataFrame(rows).T.round(3)

**Question 6.** (a) Which polygons change most between dates, and what are they likely to be on the ground? (b) Why do the January and May features *lower* the accuracy of predicting these labels? (c) For which kind of legend would temporal features be essential?

*Your answer:*

- **(a) Polygons with largest temporal change:** Polygons classified as **bare soil** (e.g., `poly_44`, `poly_39`) and **sparse vegetation** display dramatic seasonal NDVI shifts, surging from low values (~0.1) in September up to >0.5–0.7 in January and May. On the ground, these correspond to **seasonal croplands and agricultural fields** that are cultivated and green during the rainy seasons (Season A and B) but harvested, cleared, or dry during the September dry season.
- **(b) Why multi-date features reduce accuracy for these labels:** The reference labels were annotated strictly based on the September single-date snapshot. Including wet-season (Jan/May) features reveals high greenness in areas labeled as "bare soil", causing the classifier to confuse seasonal fields with permanent vegetation and lowering accuracy against dry-season labels.
- **(c) When temporal features are essential:** Multi-temporal features are vital for **land-use and crop-type classification legends**—such as differentiating annual crops from permanent bare ground/built-up surfaces, identifying single- vs double-cropped land, and distinguishing deciduous forests from evergreen canopies.

## Step 9 · Forward feature selection with spatial CV  *(LO7)*

`SequentialFeatureSelector` adds, one at a time, the feature that most improves the cross-validated accuracy. We give it our spatial folds (≈2–4 minutes).

In [ ]:
from sklearn.feature_selection import SequentialFeatureSelector
sfs = SequentialFeatureSelector(RandomForestClassifier(n_estimators=60, random_state=0, n_jobs=-1),
                                n_features_to_select=4, direction="forward", cv=cv, scoring="accuracy")
sfs.fit(X[ALL].values, y)
chosen = list(np.array(ALL)[sfs.get_support()])
print("selected:", chosen)
pd.DataFrame({"4 selected features": evaluate(rf, chosen), "all 27 features": evaluate(rf, ALL)}).T.round(3)

**Question 7.** Compare the four selected features with all 24. Why is the accuracy of the selected set optimistic, and how would nested cross-validation give an honest estimate?

*Your answer:*

- **Comparison:** Sequential Forward Selection (SFS) selects a compact 4-feature subset that achieves accuracy equal to or slightly higher than using all 24/27 features, confirming that a few well-chosen spectral dimensions suffice for land-cover mapping.
- **Why the selected accuracy is optimistic:** Feature selection was performed on the same cross-validation splits used to report the final evaluation score. The selection algorithm evaluated many feature combinations and selected the subset that performed best on those specific validation folds (selection bias / meta-overfitting).
- **How nested cross-validation resolves this:** Nested cross-validation implements two separate loops:
  1. An **inner loop** that performs feature selection strictly on the training folds.
  2. An **outer loop** that evaluates the selected model on an independent, held-out test fold that was never involved in the feature selection process.
  This guarantees an unbiased, leakage-free estimate of generalization performance.

## Reflection 

1. Which feature set would you choose for a Rwanda-wide land-cover map, considering accuracy, cost, interpretability and transfer to other dates? Justify with results from this notebook.
2. Name two things you learned about the Kigali teaching labels from the January and May images, and how you would improve the labels.
3. Which transformations in this notebook must be fitted on training data only, and which were fitted without labels (and why is that acceptable)?

*Your answer:*

1. **Optimal feature set for Rwanda-wide mapping:**
   - Select a concise, physically grounded feature set: the primary spectral bands (B02–B04, B08, B11, B12), plus 2–3 non-redundant indices (e.g., NDVI or NDRE for vegetation, MNDWI for water, NDBI for built-up) and multi-temporal NDVI metrics (min, max, amplitude) if mapping agricultural land use.
   - *Justification:* Physical indices are computationally efficient to scale across Rwanda in Google Earth Engine, easy to interpret, and transfer reliably across scenes without the scene-specific distortions of PCA.
2. **Lessons from multi-date imagery regarding Kigali teaching labels:**
   - Several September "bare soil" and "built-up" polygons are in reality seasonal agricultural fields that green up during wet months.
   - *How to improve labels:* Define whether the legend targets instantaneous land cover (on an exact acquisition date) or annual land use; validate reference sites using multi-temporal profiles or high-resolution seasonal field inspections.
3. **Fitting rules and data leakage:**
   - **Must be fitted on training folds only (inside pipeline):** `StandardScaler`, `PCA` when used as classifier input features, and `SequentialFeatureSelector`. Fitting them on the full dataset leaks validation fold distribution parameters and class labels.
   - **Fitted without labels on full scene:** Scene-wide PCA for visual composites and correlation matrices. Because these transformations are completely unsupervised and do not utilize class labels, they do not directly leak target information, though for strict spatial evaluation all feature transforms should ideally be restricted to training areas.